In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Path to dataset files:", path)

Path to dataset files: /home/nikita/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


In [3]:
import os

file = os.listdir(path)
files_in_dir = os.listdir(path + '/' + file[0])

print(files_in_dir)

path_to_files = path + '/' + file[0]

['genres_original', 'features_3_sec.csv', 'features_30_sec.csv', 'images_original']


In [4]:
base_path = path_to_files + '/' + files_in_dir[3]

files_list = os.listdir(base_path)

data_files = []

for i in range(len(files_list)):
    class_path = os.path.join(base_path, files_list[i])
    path_obj = os.listdir(class_path)
    path_obj_full = [os.path.join(class_path, f) for f in path_obj]
    data_files.append(path_obj_full)

In [10]:
import pandas as pd
import numpy as np
from PIL import Image
import random

data_x = []
data_y = []
size = (260, 260)

for class_idx, file_list in enumerate(data_files):
    num_samples = len(file_list)
    
    for j in range(num_samples):
        img = Image.open(file_list[j])
        res = img.convert('RGB').resize(size)
        res = np.array(res) / 255.0
        v_stripes = np.hsplit(res, 10)
        data_x.append(v_stripes)
        data_y.append(class_idx)

In [12]:
from sklearn.model_selection import train_test_split

data_x = np.array(data_x)
data_y = np.array(data_y)

x_train, x_test, y_train, y_test = train_test_split(
    data_x, 
    data_y, 
    test_size=0.10, 
    random_state=42,
    stratify=data_y
)

In [13]:
x_train.shape

(899, 10, 260, 26, 3)

In [14]:
del data_x
del data_y

In [28]:
import tensorflow as tf
import keras
from tensorflow.keras.callbacks import EarlyStopping
from keras.layers import Conv2D, MaxPooling2D, Flatten
from tensorflow.keras import layers

print("Версия TensorFlow:", tf.__version__)
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("Устройство для вычислений:", tf.test.is_gpu_available())

Версия TensorFlow: 2.14.0
Доступные GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Устройство для вычислений: True


2026-05-04 20:13:32.091109: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-04 20:13:32.093230: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-04 20:13:32.094933: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [51]:
model = keras.Sequential([
    keras.Input(shape = (10, 260, 26, 3)),
    keras.layers.TimeDistributed(Conv2D(5, (3, 3), padding = 'same', strides = 2)),
    keras.layers.Dropout(0.25),
    keras.layers.TimeDistributed(MaxPooling2D((2, 4))),
    keras.layers.TimeDistributed(Conv2D(5, (3, 3), padding = 'same', activation ='relu', strides = 2)),
    keras.layers.Dropout(0.25),
    keras.layers.TimeDistributed(Flatten()),
    keras.layers.GRU(32, return_sequences=False, activation='tanh'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation ='softmax')
])

model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-4),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    patience = 50,
                    min_delta = 0.01,
                    verbose = 0,
                    restore_best_weights = True
                    )

In [52]:
model.fit(
        x = x_train,
        y = y_train,
        batch_size = 8,
        epochs = 2000,
        verbose = 0,
        callbacks = [early_stopping], 
        shuffle = True,
        validation_split = 0.1,
        validation_data = None,
        validation_batch_size = 8
        )

In [53]:
train_loss, keras_train_acc = model.evaluate(x_train, y_train)
test_loss, keras_test_acc = model.evaluate(x_test, y_test)
print('\nдля CNN+RNN точность для обучающей \ тестовой выборки:', round(keras_train_acc, 2), ' \ ', round(keras_test_acc, 2))

4/4 [==============================] - 0s 5ms/step - loss: 1.2283 - accuracy: 0.6000

для CNN+RNN точность для обучающей \ тестовой выборки: 0.83  \  0.6


In [54]:
from tensorflow.keras.layers import Attention

In [55]:
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, use_scale=False, **kwargs):
        super().__init__(**kwargs)
        self.use_scale = use_scale
        self.attention = Attention(use_scale=use_scale)
    
    def build(self, input_shape):
        self.W = self.add_weight(
            name='att_weight',
            shape=(input_shape[-1], 1),
            initializer='glorot_uniform',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        # x: (batch, timesteps, features)
        e = tf.matmul(x, self.W)                # (batch, timesteps, 1)
        e = tf.squeeze(e, axis=-1)              # (batch, timesteps)
        alpha = tf.nn.softmax(e, axis=-1)       # веса внимания (batch, timesteps)
        context = tf.reduce_sum(x * tf.expand_dims(alpha, -1), axis=1)  # (batch, features)
        return context

model_2 = keras.Sequential([
    keras.Input(shape = (10, 260, 26, 3)),
    keras.layers.TimeDistributed(Conv2D(5, (3, 3), padding = 'same', strides = 2)),
    keras.layers.Dropout(0.25),
    keras.layers.TimeDistributed(MaxPooling2D((2, 4))),
    keras.layers.TimeDistributed(Conv2D(5, (3, 3), padding = 'same', activation ='relu', strides = 2)),
    keras.layers.Dropout(0.25),
    keras.layers.TimeDistributed(Flatten()),
    keras.layers.GRU(32, return_sequences=True, activation='tanh'),
    SelfAttention(use_scale=True),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation ='softmax')
])

model_2.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-4),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping_2 = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    patience = 50,
                    min_delta = 0.01,
                    verbose = 0,
                    restore_best_weights = True
                    )

In [56]:
model_2.fit(
        x = x_train,
        y = y_train,
        batch_size = 8,
        epochs = 2000,
        verbose = 0,
        callbacks = [early_stopping_2], 
        shuffle = True,
        validation_split = 0.1,
        validation_data = None,
        validation_batch_size = 8
        )

2026-05-04 20:53:36.212227: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForce RTX 4070 SUPER" frequency: 2475 num_cores: 56 environment { key: "architecture" value: "8.9" } environment { key: "cuda" value: "11080" } environment { key: "cudnn" value: "8600" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 50331648 shared_memory_size_per_multiprocessor: 102400 memory_size: 10257301504 bandwidth: 504048000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2026-05-04 20:53:37.879838: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForc

In [58]:
train_loss_2, keras_train_acc_2 = model_2.evaluate(x_train, y_train)
test_loss_2, keras_test_acc_2 = model_2.evaluate(x_test, y_test)
print('\nдля CNN+RNN+Attention точность для обучающей \ тестовой выборки:', round(keras_train_acc_2, 2), ' \ ', round(keras_test_acc_2, 2))

4/4 [==============================] - 0s 4ms/step - loss: 0.9786 - accuracy: 0.6800

для CNN+RNN+Attention точность для обучающей \ тестовой выборки: 0.77  \  0.68
